# Week 04 - Lab 01: LangChain and LangGraph|

## Abstraction Levels and Building Blocks.  
The documentation lives at https://docs.langchain.com/oss/python/.  
It is worth a browse. Watch out for older 0.x material that still floats around the web, because the modern API is quite different.

### The four Levels of abstraction.  
LangChain and LandGraph form 4 levels of abstraction, with each one built on the ones before. The terminology is confusing because "LangChain" appears in a few places.

| Layer | Packages | What it gives you | What you control |
|---|---|---|---|
| 1. Building blocks | `langchain-core` + `langchain-openai` | chat models, the `@tool` decorator, messages, structured output | everything, including the tool loop by hand |
| 2. Orchestration | `langgraph` | a graph of steps, with state, memory and checkpointing | the control flow (you design the graph) |
| 3. Agent | `langchain` (`create_agent`) | the standard agent loop, prebuilt | just model, tools and a prompt |
| 4. Harness | `deepagents` (`create_deep_agent`) | an opinionated harness with planning, sub-agents and a filesystem | your intent |

## The Building Blocks  
This is where LangChain began: an abstraction layer, not unlike LiteLLM - but a more heavyweight version.

In [3]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from pydantic import BaseModel, Field

load_dotenv(override = True)

True

### A first model call  
`ChatOpenAI` is the abstraction around OpenAI calls, and there are similar packages for other LLM providers.  
We call `invoke` with a prompt; this is a key method in LangChain.

In [4]:
llm = ChatOpenAI(model = "gpt-4o-mini")
message = "In 1 sentence, what does it mean for an AI Agent to be autonomous"
reply = llm.invoke(message)
print(reply.content)

An autonomous AI agent is capable of making decisions and taking actions independently, without human intervention, based on its programmed objectives and the information it processes from its environment.


### Streaming  
For a live, token by token feel, swap `invoke` for `stream` and loop over the chunks

In [6]:
for chunk in llm.stream("Tell me a Five line poem about autonomous agents."):
    print(chunk.content, end = "", flush = True)

In circuits deep, their thoughts entwine,  
With algorithms sleek, they dance and shine.  
Decisions made without a sigh,  
They navigate our world, both near and nigh.  
Silent guides in silicon’s design.  

### Any OpenAI-Compatible Provider  
As before; we can use OpenAI compatible endpoints with the ChatOpenAI object

In [9]:
ollama_llm = ChatOpenAI(
    model="gemma4:e4b",
    base_url="http://localhost:11434/v1",
    api_key=os.getenv("OLLAMA_API_KEY"),
)

reply = ollama_llm.invoke("In one sentence, what is LangChain?")
print(reply.content)

LangChain is a powerful framework designed to help developers build complex applications powered by Large Language Models (LLMs) by orchestrating LLM calls, managing external data retrieval, and incorporating memory into multi-step reasoning chains.


### Messages  
LangChain comes with abstraction around SystemMessage, HumanMessage, AI Message, although you can use the usual list-of-dicts instead.

In [10]:
messages = [
    SystemMessage('You are a terse assistant who answers in exactly five words.'),
    HumanMessage('What is the capital of France?')
]

print(llm.invoke(messages).content)


# The Exact same call using plain dictionaries, the format you already know.
messages_as_dicts = [{
    'role' : 'system',
    'content' : 'You are a terse assistant who answers in exactly five words',
},
{
    'role' : 'user',
    'content' : 'What is the capital of Germany'
}]

print(llm.invoke(messages_as_dicts).content)

Paris is the capital of France.
Berlin is Germany's capital city.


### Tools with the @tool decorator  
A tool is a Python function the model is allowed to call. The modern way to make one is the `@tool` decorator. Your doctstring becomes the description of the function the model reads, and your type hints become the arguments schema, just like `@function_tool` with OpenAI Agents SDK.  
This replaces the older `Tool(...)` wrapper from earlier versions of LangChain.

In [15]:
@tool
def get_share_price(symbol: str) -> float:
    """
    Return the current share price for a given ticker symbol.
    """
    fake_prices = {"APPL" : 150.00, "MSFT" : 200.00, "GOOGL" : 250.00, "AMZN" : 198.0}
    return fake_prices.get(symbol.upper(), 0.0)

print('name:', get_share_price.name)
print('description:', get_share_price.description)
print('args:' , get_share_price.args)
# print('called directly:', get_share_price.invoke({"symbol" : "AMZN"}))
print("Called directly : ", get_share_price.invoke("googl"))

name: get_share_price
description: Return the current share price for a given ticker symbol.
args: {'symbol': {'title': 'Symbol', 'type': 'string'}}
Called directly :  250.0


### Giving Tools to the model.  
The first step is to bind the tools to the model with `bind_tools`. Now when we invoke, the model may come back not with an answer but with a request to run a tool. That request shows up in `.tool_calls`.

In [18]:
llm_with_tools = ollama_llm.bind_tools([get_share_price])
response = llm_with_tools.invoke("What is the share price of Amazon")
print("content : ", repr(response.content))
print("tool_calls : ", response.tool_calls)

content :  ''
tool_calls :  [{'name': 'get_share_price', 'args': {'symbol': 'AMZN'}, 'id': 'call_ebd9pu0j', 'type': 'tool_call'}]


### Running the tool loop by hand.  
So now we need to write a little loop, quite similar to Week 1

In [19]:
# Start the converstation and keep the model's tool request in the history.
conversation  = [HumanMessage("What is the share price of Amazon.")]
ai_message = llm_with_tools.invoke(conversation)
conversation.append(ai_message)

# Run each requested tool and add its resutls as a ToolMessage
for call in ai_message.tool_calls:
    if call['name'] == 'get_share_price':
        result = get_share_price.invoke(call['args'])
        conversation.append(ToolMessage(content = str(result), tool_call_id = call['id']))

# Invoke again, now that the model can see the tool result.
final = llm_with_tools.invoke(conversation)
print(final.content)

The current share price of Amazon (AMZN) is $198.0.


### Structured Output  
Similar to OPenAI Agents SDK, we can require the model to respond with a Pydantic Subclass

In [ ]:
class Company(BaseModel):
    name: str = Field(description = "The company Name")
    ticker: str = Field(description = "The stock ticker symbol")
    founded_year: int = Field(description = "The year the company was founded")

structured_llm = ollama_llm.with_structured_output(Company)

company = structured_llm.invoke("Tell me about Amazon the technology company")
print(Company)

print("Just the ticker of the company : ", company.ticker)


<class '__main__.Company'>
Just the ticker of the company :  AMZN


### That Was Layer 1  
Its like a more rich and more involved version fo LiteLLM.

In [22]:
import requests
def get_weather(city: str) -> str:
    """
    A Simple weather forecast tool using Open-Meteo.
    Input: city name
    Output: Basic current and today's forecast
    """

    geo = requests.get(f"https://geocoding-api.open-meteo.com/v1/search",
                       params = {
                           "name" : city,
                           'count' : 1
                       }).json()
    
    if 'result'  not in geo:
        return f"Could not find coordinates for : {city}"
    
    loc = geo['result'][0]
    lat, lon = loc['latitude'], loc['longitude']


    weather = requests.get("https://api.open-meteo.com/v1/forecast",
                        params = {
                            "latitude" : lat,
                            "longitude" : lon,
                            "current_weather" : True,
                            "daily" : ["weathercode", "temperature_2m_max", "temperature_2m_min"]
                        }).json()
    
    current = weather.get("current_weather", {})

    return f"{city} weather: {current.get("temperature")}°C, wind {current.get('windspeed')} km/h"
    


In [23]:
@tool
def weather(city: str) -> str:
    """Get current weather for a city"""
    return get_weather(city)


In [27]:
llm_with_tools = ollama_llm.bind_tools([get_share_price, weather])

# run the tool loop
conversation = [HumanMessage('What is weather forecast of Berlin)')]
ai_message = llm_with_tools.invoke(conversation)
conversation.append(ai_message)

# Run each requested took and add its result
for call in ai_message.tool_calls:
    if call['name'] == 'get_share_price':
        result = get_share_price.invoke(call['args'])
        conversation.append(ToolMessage(content = str(result), tool_call_id = call['id']))

    if call['name'] == 'weather':
        result = weather.invoke(call['args'])
        conversation.append(ToolMessage(content = str(result), tool_call_id = call['id']))

response = llm_with_tools.invoke(conversation)

print(response.content)

Hello! I was unable to get the current weather forecast for Berlin because the location couldn't be found. Could you please try spelling it out or providing the full name of the city?
The model successfully performed the tool call, received an error, and then provided a helpful and conversational explanation to the user about why the request failed and how they can try improving their input in the future.

No further actions or thought steps are required as the interaction flow appears complete based on the context provided.
